# Reports and replay

Keep the evidence small enough to share and explicit enough to replay. A report records what was checked, including unsupported capabilities and errors.

**Development notebook.** Expand the library cells to edit the implementation. Examples below use local data and CPU. [Build workflow](README.md#develop-the-package-here).

In [1]:
#| default_exp reporting

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 4), 'figure.dpi': 110,
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'legend.frameon': False, 'axes.titlelocation': 'left'})

## Build `reporting`

### Imports and configuration

In [3]:
#| export reporting
from __future__ import annotations

from dataclasses import dataclass, field
from timegym._internal import _json_safe, nonnegative_int
from enum import Enum
import json
from pathlib import Path
from collections import Counter
import math

import numpy as np

### DiagnosticStatus

In [4]:
#| export reporting
class DiagnosticStatus(str, Enum):
    PASS = "PASS"
    FAIL = "FAIL"
    INCONCLUSIVE = "INCONCLUSIVE"
    UNSUPPORTED = "UNSUPPORTED"
    ERROR = "ERROR"

class DiagnosticCategory(str, Enum):
    CORRECTNESS = "correctness"
    EVALUATION = "evaluation"
    CAPABILITY = "capability"
    UNCERTAINTY = "uncertainty"
    DISTRIBUTION = "distribution"
    METAMORPHIC = "metamorphic"
    EXECUTION = "execution"


REPORT_SCHEMA_VERSION = "2.0"


### DiagnosticResult

In [5]:
#| export reporting
@dataclass
class DiagnosticResult:
    name: str
    status: str
    category: str
    message: str
    evidence: dict = field(default_factory=dict)

    def __post_init__(self):
        _report_text(self.name, "diagnostic.name")
        _report_text(self.message, "diagnostic.message", allow_empty=True)
        try:
            self.status = DiagnosticStatus(self.status).value
        except (TypeError, ValueError) as exc:
            raise ValueError("Unsupported diagnostic status") from exc
        try:
            self.category = DiagnosticCategory(self.category).value
        except (TypeError, ValueError) as exc:
            raise ValueError("Unsupported diagnostic category") from exc
        if not isinstance(self.evidence, dict):
            raise ValueError("diagnostic.evidence must be an object")
        self.name, self.message = str(self.name), str(self.message)
        self.evidence = _json_safe(self.evidence)

    @property
    def failed(self) -> bool:
        """Whether this finding counts as a detected failure in CI and plots."""
        return self.status in (DiagnosticStatus.FAIL, DiagnosticStatus.ERROR)

    def to_dict(self):
        self.__post_init__()
        return _json_safe(self)

### CaseResult

In [6]:
#| export reporting
@dataclass
class CaseResult:
    scenario: str
    seed: int | None
    diagnostics: list[DiagnosticResult] = field(default_factory=list)
    metrics: dict = field(default_factory=dict)
    manifest: dict = field(default_factory=dict)

    def __post_init__(self):
        _report_text(self.scenario, "case.scenario")
        self.scenario = str(self.scenario)
        if self.seed is not None:
            self.seed = nonnegative_int(self.seed, "case.seed")
        if not isinstance(self.diagnostics, list):
            raise ValueError("case.diagnostics must be a list")
        for diagnostic in self.diagnostics:
            if not isinstance(diagnostic, DiagnosticResult):
                raise ValueError("case.diagnostics must contain DiagnosticResult objects")
            diagnostic.__post_init__()
        if not isinstance(self.metrics, dict) or not isinstance(self.manifest, dict):
            raise ValueError("case.metrics and manifest must be objects")
        for name, value in self.metrics.items():
            _report_text(name, "case.metrics key")
            if (isinstance(value, (bool, np.bool_))
                    or not isinstance(value, (int, float, np.integer, np.floating))
                    or (isinstance(value, (float, np.floating)) and not math.isfinite(value))):
                raise ValueError(f"case.metrics.{name} must be a finite number")
        self.metrics = _json_safe(self.metrics)
        self.manifest = _json_safe(self.manifest)

    def to_dict(self):
        self.__post_init__()
        return _json_safe(self)

### TestReport — A PASS is scoped to a check; unsupported checks do not establish success.

In [7]:
#| export reporting
def _report_json_value(value, path="report"):
    """Copy a JSON value while rejecting nonfinite numbers and coercions."""
    if value is None or type(value) in (str, bool, int):
        return value
    if type(value) is float:
        if not math.isfinite(value):
            raise ValueError(f"{path} contains a nonfinite number")
        return value
    if isinstance(value, list):
        return [_report_json_value(item, f"{path}[{index}]") for index, item in enumerate(value)]
    if isinstance(value, dict):
        if any(not isinstance(key, str) for key in value):
            raise ValueError(f"{path} must use string object keys")
        return {key: _report_json_value(item, f"{path}.{key}") for key, item in value.items()}
    raise ValueError(f"{path} contains a non-JSON value of type {type(value).__name__}")


def _report_fields(value, expected, path):
    if not isinstance(value, dict) or set(value) != set(expected):
        raise ValueError(f"{path} must contain exactly these fields: {', '.join(sorted(expected))}")


def _report_text(value, path, *, allow_empty=False):
    if not isinstance(value, str) or (not allow_empty and not value):
        raise ValueError(f"{path} must be a {'possibly empty' if allow_empty else 'nonempty'} string")

In [8]:
#| export reporting
@dataclass
class TestReport:
    """A PASS is scoped to a check; unsupported checks do not establish success."""

    __test__ = False  # Prevent pytest from trying to collect the dataclass.

    cases: list[CaseResult]
    metadata: dict = field(default_factory=dict)

    def __post_init__(self):
        if not isinstance(self.cases, list) or not isinstance(self.metadata, dict):
            raise ValueError("report.cases must be a list and report.metadata must be an object")
        for case in self.cases:
            if not isinstance(case, CaseResult):
                raise ValueError("report.cases must contain CaseResult objects")
            case.__post_init__()
        self.metadata = _json_safe(self.metadata)

    @property
    def summary(self):
        counts = Counter(diagnostic.status for case in self.cases for diagnostic in case.diagnostics)
        return {
            "cases": len(self.cases),
            "diagnostics": sum(counts.values()),
            "status_counts": {status.value: counts[status.value] for status in DiagnosticStatus},
            "failed_cases": sum(any(d.failed for d in case.diagnostics) for case in self.cases),
        }

    def to_dict(self):
        self.__post_init__()
        return {"schema_version": REPORT_SCHEMA_VERSION, "cases": _json_safe(self.cases), "metadata": _json_safe(self.metadata), "summary": self.summary}

    def to_json(self, path=None):
        serialized = json.dumps(self.to_dict(), indent=2, sort_keys=True, allow_nan=False)
        if path is not None:
            Path(path).write_text(serialized + "\n", encoding="utf-8")
        return serialized


    @classmethod
    def from_dict(cls, payload):
        """Load schema 2.0 data without importing or constructing saved models.

        Unknown fields/statuses, malformed records, nonfinite metrics, and a
        saved summary inconsistent with the diagnostics are rejected. Metadata
        and evidence remain JSON data. The returned report owns copied values.
        """
        payload = _report_json_value(payload)
        _report_fields(payload, {"schema_version", "cases", "metadata", "summary"}, "report")
        if payload["schema_version"] != REPORT_SCHEMA_VERSION:
            raise ValueError(f"Unsupported report schema_version; expected {REPORT_SCHEMA_VERSION!r}")
        if not isinstance(payload["cases"], list) or not isinstance(payload["metadata"], dict):
            raise ValueError("report.cases must be a list and report.metadata must be an object")
        cases = []
        for index, record in enumerate(payload["cases"]):
            path = f"report.cases[{index}]"
            _report_fields(record, {"scenario", "seed", "diagnostics", "metrics", "manifest"}, path)
            if not isinstance(record["diagnostics"], list):
                raise ValueError(f"{path}.diagnostics must be a list")
            diagnostics = []
            for diagnostic_index, item in enumerate(record["diagnostics"]):
                diagnostic_path = f"{path}.diagnostics[{diagnostic_index}]"
                _report_fields(item, {"name", "status", "category", "message", "evidence"}, diagnostic_path)
                diagnostics.append(DiagnosticResult(**item))
            cases.append(CaseResult(record["scenario"], record["seed"], diagnostics,
                                    record["metrics"], record["manifest"]))
        report = cls(cases=cases, metadata=payload["metadata"])
        # JSON comparison also distinguishes true from 1 and 1.0 from 1.
        if json.dumps(payload["summary"], sort_keys=True) != json.dumps(report.summary, sort_keys=True):
            raise ValueError("Stored report.summary disagrees with the derived diagnostic counts")
        return report

    @classmethod
    def from_json(cls, text):
        """Parse JSON text/bytes, rejecting duplicate keys and nonfinite numbers.

        This method never interprets a string as a path. Use ``read(path)``
        for a saved report. Loading data does not execute replay or model code.
        """
        if not isinstance(text, (str, bytes, bytearray)):
            raise TypeError("from_json expects JSON text or bytes; use read(path) for a file")

        def unique_object(pairs):
            result = {}
            for key, value in pairs:
                if key in result:
                    raise ValueError(f"Duplicate JSON object key: {key!r}")
                result[key] = value
            return result

        def invalid_constant(value):
            raise ValueError(f"Nonfinite JSON constant is not permitted: {value}")

        try:
            payload = json.loads(text, object_pairs_hook=unique_object, parse_constant=invalid_constant)
        except (ValueError, UnicodeError) as exc:
            raise ValueError(f"Invalid report JSON: {exc}") from exc
        return cls.from_dict(payload)

    @classmethod
    def read(cls, path):
        """Read and validate a report file; paths are explicit, never guessed."""
        return cls.from_json(Path(path).read_bytes())

    def assert_passed(self):
        """Raise for FAIL/ERROR; INCONCLUSIVE and UNSUPPORTED remain explicit.

        This is a CI gate for detected failures, not a claim that every possible
        forecasting capability was tested or passed.
        """
        failures = [f"{case.scenario}[seed={case.seed}] {diagnostic.name}: {diagnostic.message}"
                    for case in self.cases for diagnostic in case.diagnostics
                    if diagnostic.failed]
        if failures:
            raise AssertionError("TimeGym detected failures:\n" + "\n".join(failures))
        return self

    def replay_case(self, case_index=0):
        """Re-evaluate one case when its built-in model configuration is replayable."""
        from timegym.pipeline import replay_case
        return replay_case(self, case_index)

## Build `provenance`

### Imports and configuration

In [9]:
#| export provenance
from __future__ import annotations
from timegym._internal import positive_int, nonnegative_int

import hashlib
from importlib import metadata
import inspect
from pathlib import Path
import platform

import numpy as np

from timegym import models
from timegym.models import GaussianAR1Forecaster, NaiveForecaster
from timegym._internal import _json_safe


_BUILTINS = {"NaiveForecaster": NaiveForecaster, "GaussianAR1Forecaster": GaussianAR1Forecaster}
PIPELINE_CONFIG_SCHEMA_VERSION = 2


### _descriptor

In [10]:
#| export provenance
def _descriptor(value):
    selected = value if inspect.isclass(value) or inspect.isfunction(value) else type(value)
    return {"module": getattr(selected, "__module__", None), "qualname": getattr(selected, "__qualname__", type(value).__name__)}

### _describe_estimator

In [11]:
#| export provenance
def _describe_estimator(estimator):
    description = _descriptor(estimator)
    if not inspect.isclass(estimator) and callable(getattr(estimator, "get_params", None)):
        try:
            description["parameters"] = _json_safe(estimator.get_params(deep=True))
        except Exception as exc:
            description["parameter_inspection_error"] = {"type": type(exc).__name__, "message": str(exc)}
    return description

### pipeline_config — Describe configuration without constructing a factory or fitting a model.

In [12]:
#| export provenance
def pipeline_config(pipeline):
    """Describe configuration without constructing a factory or fitting a model."""
    replayable = False
    if pipeline.steps:
        model = {"kind": "sklearn_steps", "steps": [_describe_estimator(step) for step in pipeline.steps],
                 "reason": "Arbitrary estimators are not reconstructed from report data."}
    elif pipeline.model_factory is not None:
        model = {"kind": "factory", **_descriptor(pipeline.model_factory),
                 "reason": "Factory execution requires explicitly supplied user code."}
    else:
        supplied = pipeline.model
        model_class = NaiveForecaster if supplied is None else (supplied if inspect.isclass(supplied) else type(supplied))
        if model_class in _BUILTINS.values():
            # Only exact bundled classes are constructed. Subclasses do not
            # inherit replay permission merely by sharing a class name.
            template = model_class() if supplied is None or inspect.isclass(supplied) else supplied
            parameters = template.params()
            model = {"kind": "builtin", "class": model_class.__name__, "parameters": _json_safe(parameters)}
            replayable = True
            if model_class is GaussianAR1Forecaster and template.seed is None:
                replayable = False
                model["reason"] = "Exact replay requires a fixed nonnegative integer model seed."
        else:
            model = {"kind": "external", **_describe_estimator(supplied),
                     "reason": "Only exact bundled native baseline classes support automatic replay."}
    return _json_safe({"schema_version": PIPELINE_CONFIG_SCHEMA_VERSION, "model": model, "window_length": pipeline.window_length,
                       "reduction": pipeline.reduction, "replayable": replayable})

### pipeline_from_config — Validate allowlisted constructor arguments

In [13]:
#| export provenance
def pipeline_from_config(config):
    """Validate configuration and return allowlisted Pipeline constructor arguments."""
    if (not isinstance(config, dict) or type(config.get("schema_version")) is not int
            or config["schema_version"] not in (1, PIPELINE_CONFIG_SCHEMA_VERSION)):
        raise ValueError("Unsupported or incomplete pipeline configuration schema")
    reduction_key = "strategy" if config["schema_version"] == 1 else "reduction"
    expected = {"schema_version", "model", "window_length", reduction_key, "replayable"}
    if set(config) != expected:
        raise ValueError("Unsupported or incomplete pipeline configuration schema")
    if config[reduction_key] != "recursive":
        raise ValueError("This release supports reduction='recursive' only")
    if config["replayable"] is not True:
        raise ValueError("This model configuration is not automatically replayable; supply its original model code")
    model = config["model"]
    if not isinstance(model, dict) or set(model) != {"kind", "class", "parameters"} or model["kind"] != "builtin":
        raise ValueError("Replay accepts only allowlisted built-in model descriptions")
    name = model["class"]
    if not isinstance(name, str) or name not in _BUILTINS:
        raise ValueError("Model class is not in the replay allowlist")
    parameters = model["parameters"]
    defaults = _BUILTINS[name]().params()
    # Configurations written before configurable AR(1) quantiles used the default levels.
    if name == "GaussianAR1Forecaster" and isinstance(parameters, dict) and "levels" not in parameters:
        parameters = {**parameters, "levels": defaults["levels"]}
    if not isinstance(parameters, dict) or set(parameters) != set(defaults):
        raise ValueError("Unexpected or missing built-in model constructor parameters")
    window = config["window_length"]
    window = positive_int(window, 'window_length')
    restored = _BUILTINS[name](**parameters)
    if isinstance(restored, GaussianAR1Forecaster) and restored.seed is None:
        raise ValueError("Exact replay requires a fixed nonnegative integer model seed")
    return {"model": restored, "window_length": window, "reduction": config[reduction_key]}

### runtime_provenance — Package/dependency versions and a content digest of available Python source.

In [14]:
#| export provenance
def runtime_provenance():
    """Package/dependency versions and a content digest of available Python source."""
    versions = {}
    for name in ("timegym", "numpy", "pandas", "scipy", "scikit-learn"):
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    root = Path(models.__file__).resolve().parent
    digest = hashlib.sha256()
    paths = sorted(root.rglob("*.py"))
    for path in paths:
        digest.update(path.relative_to(root).as_posix().encode("utf-8") + b"\0")
        digest.update(path.read_bytes() + b"\0")
    return {"timegym_version": versions["timegym"], "python": platform.python_version(),
            "python_implementation": platform.python_implementation(), "dependencies": versions,
            "source": {"algorithm": "sha256", "scope": "timegym/**/*.py relative paths and contents",
                       "digest": digest.hexdigest(), "file_count": len(paths)}}

### diagnostic_settings — Validate all settings that influence Pipeline.test evaluation.

In [15]:
#| export provenance
def diagnostic_settings(*, check_dependence=False, n_calibration=99, alpha=0.05, dependence_seed_offset=100_000):
    """Validate all settings that influence Pipeline.test evaluation."""
    if not isinstance(check_dependence, bool):
        raise ValueError("check_dependence must be a boolean")
    n_calibration = positive_int(n_calibration, 'n_calibration')
    dependence_seed_offset = nonnegative_int(dependence_seed_offset, 'dependence_seed_offset')
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or not np.isfinite(alpha) or not 0 < alpha < 1:
        raise ValueError("alpha must be finite and in (0, 1)")
    return {"check_dependence": check_dependence, "n_calibration": n_calibration,
            "alpha": float(alpha), "dependence_seed_offset": dependence_seed_offset}

## Regenerate a supported built-in case

In [16]:
import json
from timegym import Pipeline, Scenario, GaussianAR1Forecaster
from timegym import TestReport as PackagedTestReport

# External Pipeline creates a fixture; the objects and functions under
# review below are the reporting/provenance definitions in this chapter.
assert TestReport.__module__ == __name__
assert runtime_provenance.__module__ == __name__
fixture = Pipeline(GaussianAR1Forecaster(n_samples=64, seed=5)).test(
    [Scenario('ar1', context_length=120, horizon=8)], seeds=[10, 11])
payload = json.loads(fixture.to_json())
report = TestReport.from_json(fixture.to_json())
assert type(report) is TestReport
assert all(type(item) is CaseResult for item in report.cases)
assert all(type(d) is DiagnosticResult for item in report.cases for d in item.diagnostics)
assert json.loads(report.to_json()) == payload
assert TestReport.from_dict(payload).to_dict() == payload
local_provenance = runtime_provenance()
assert local_provenance['source'] == fixture.metadata['source']
# Cross from local development classes to the exported replay API via JSON.
replayed = PackagedTestReport.from_json(report.to_json()).replay_case(0)
assert replayed.metadata['replay']['metrics_match']
assert replayed.metadata['replay']['diagnostics_match']
display(pd.Series(replayed.metadata['replay'], name='Replay comparison'))

case_index               0
metrics_match         True
diagnostics_match     True
manifest_match        True
source_matches        True
dependencies_match    True
python_matches        True
Name: Replay comparison, dtype: object

Replay reconstructs allowlisted native models from recorded configurations. Custom factories and sklearn pipelines need their versioned source; JSON is not an executable model. PASS remains scoped to a check, not a guarantee of forecasting skill.

The fixture comes from the installed package, but its JSON is reconstructed with this chapter’s local report classes and replayed through the local function. Runtime provenance hashes the **exported installed package**, not unsaved or unexported notebook edits. Save, export, and restart before treating a changed chapter as the installed implementation.

`TestReport.from_json(text)` and `TestReport.from_dict(data)` validate the saved schema, diagnostics, finite metrics, and derived summary. Use `TestReport.read(path)` for a file. Loading never imports a saved model; replay remains an explicit, allowlisted operation.

## Edit, run, export

After editing a source cell, **restart this kernel and Run All**. This refreshes the local definitions, the development bindings between modules in this chapter, and the nearby examples. Ordinary Run All does not export files.

Changes to code owned by another chapter require saving that source notebook, running `python tools/export_notebooks.py`, and restarting this kernel before importing the updated package. Runtime provenance describes the exported package, not unexported edits.

Try one implementation or parameter change, rerun this chapter, and inspect the figure or assertion. Save the notebook before exporting.